# Celebal Technologies – Week 7 Assignment

## Delta Lake MERGE Implementation using PySpark

### Submitted By

**Amit Singh**

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta.tables import *

spark = SparkSession.builder \
    .appName("Celebal Week 7 Assignment") \
    .getOrCreate()

###Read Master Dataset

In [0]:
master_df = spark.read.csv(

"/Volumes/workspace/default/delta_lake/customer_master.csv",

header=True,

inferSchema=True

)

In [0]:
master_df.show(10)

+-----------+-------------+----------------+---------+----------+----------+--------+
|Customer_ID|Customer_Name|           Email|     City|     Phone| Join_Date|  Status|
+-----------+-------------+----------------+---------+----------+----------+--------+
|        101|   Amit Singh|  amit@gmail.com|   Jaipur|9876543210|2024-01-15|  Active|
|        102| Rahul Sharma| rahul@gmail.com|    Delhi|9876543211|2024-02-01|  Active|
|        103|  Priya Gupta| priya@gmail.com|   Mumbai|9876543212|2024-03-10|  Active|
|        104|   Neha Verma|  neha@gmail.com|     Pune|9876543213|2024-03-15|Inactive|
|        105|  Karan Patel| karan@gmail.com|Ahmedabad|9876543214|2024-04-20|  Active|
|        106|  Sneha Joshi| sneha@gmail.com|   Indore|9876543215|2024-05-12|  Active|
|        107|  Rohan Mehta| rohan@gmail.com|  Lucknow|9876543216|2024-06-05|  Active|
|        108|Anjali Sharma|anjali@gmail.com|   Bhopal|9876543217|2024-06-20|Inactive|
|        109|   Vikas Jain| vikas@gmail.com|     Kota|

In [0]:
master_df.printSchema()

root
 |-- Customer_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Status: string (nullable = true)



In [0]:
print("Total Customers :", master_df.count())

Total Customers : 20


In [0]:
master_df.columns

['Customer_ID',
 'Customer_Name',
 'Email',
 'City',
 'Phone',
 'Join_Date',
 'Status']

In [0]:
master_df.describe().show()

+-------+-----------------+-------------+------------------+---------+-----------------+--------+
|summary|      Customer_ID|Customer_Name|             Email|     City|            Phone|  Status|
+-------+-----------------+-------------+------------------+---------+-----------------+--------+
|  count|               20|           20|                20|       20|               20|      20|
|   mean|            110.5|         NULL|              NULL|     NULL|   9.8765432195E9|    NULL|
| stddev|5.916079783099616|         NULL|              NULL|     NULL|5.916079783099616|    NULL|
|    min|              101|Abhishek Jain|abhishek@gmail.com|Ahmedabad|       9876543210|  Active|
|    max|              120|   Vikas Jain|   vikas@gmail.com|  Udaipur|       9876543229|Inactive|
+-------+-----------------+-------------+------------------+---------+-----------------+--------+



The master dataset has been successfully loaded into a Spark DataFrame.



###Read Incremental Dataset

In [0]:
increment_df = spark.read.csv(

"/Volumes/workspace/default/delta_lake/customer_incremental.csv",

header=True,

inferSchema=True

)

In [0]:
increment_df.show()

+-----------+---------------+-------------------+---------+----------+----------+--------+
|Customer_ID|  Customer_Name|              Email|     City|     Phone| Join_Date|  Status|
+-----------+---------------+-------------------+---------+----------+----------+--------+
|        102|   Rahul Sharma|    rahul@gmail.com|    Noida|9876543211|2024-02-01|  Active|
|        104|     Neha Verma|     neha@gmail.com|Hyderabad|9876543213|2024-03-15|  Active|
|        115|   Deepak Yadav|   deepak@gmail.com|  Gurgaon|9876543224|2024-10-05|  Active|
|        121|    Sahil Verma|    sahil@gmail.com|     Pune|9876543230|2025-01-05|  Active|
|        122|    Ayesha Khan|   ayesha@gmail.com|   Mumbai|9876543231|2025-01-10|  Active|
|        123|   Mohit Sharma|    mohit@gmail.com|   Jaipur|9876543232|2025-01-15|  Active|
|        124|Priyanshi Gupta|priyanshi@gmail.com|    Delhi|9876543233|2025-01-20|Inactive|
|        125|    Harsh Patel|    harsh@gmail.com|Ahmedabad|9876543234|2025-02-01|  Active|

In [0]:
print("Incremental Customers :", increment_df.count())

Incremental Customers : 8


The incremental dataset represents newly arrived customer records.


##Create Delta Table

In [0]:
master_df.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/workspace/default/delta_lake/customer_delta")

The master customer dataset has been stored as a Delta Table.

In [0]:
delta_df = spark.read.format("delta").load(
"/Volumes/workspace/default/delta_lake/customer_delta"
)

In [0]:
delta_df.show()

+-----------+-------------+------------------+----------+----------+----------+--------+
|Customer_ID|Customer_Name|             Email|      City|     Phone| Join_Date|  Status|
+-----------+-------------+------------------+----------+----------+----------+--------+
|        101|   Amit Singh|    amit@gmail.com|    Jaipur|9876543210|2024-01-15|  Active|
|        102| Rahul Sharma|   rahul@gmail.com|     Delhi|9876543211|2024-02-01|  Active|
|        103|  Priya Gupta|   priya@gmail.com|    Mumbai|9876543212|2024-03-10|  Active|
|        104|   Neha Verma|    neha@gmail.com|      Pune|9876543213|2024-03-15|Inactive|
|        105|  Karan Patel|   karan@gmail.com| Ahmedabad|9876543214|2024-04-20|  Active|
|        106|  Sneha Joshi|   sneha@gmail.com|    Indore|9876543215|2024-05-12|  Active|
|        107|  Rohan Mehta|   rohan@gmail.com|   Lucknow|9876543216|2024-06-05|  Active|
|        108|Anjali Sharma|  anjali@gmail.com|    Bhopal|9876543217|2024-06-20|Inactive|
|        109|   Vikas

In [0]:
delta_df.printSchema()

root
 |-- Customer_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Status: string (nullable = true)



# Data Cleaning

Data cleaning is an important step before loading data into a Delta table.

In [0]:
master_df.show(10)

+-----------+-------------+----------------+---------+----------+----------+--------+
|Customer_ID|Customer_Name|           Email|     City|     Phone| Join_Date|  Status|
+-----------+-------------+----------------+---------+----------+----------+--------+
|        101|   Amit Singh|  amit@gmail.com|   Jaipur|9876543210|2024-01-15|  Active|
|        102| Rahul Sharma| rahul@gmail.com|    Delhi|9876543211|2024-02-01|  Active|
|        103|  Priya Gupta| priya@gmail.com|   Mumbai|9876543212|2024-03-10|  Active|
|        104|   Neha Verma|  neha@gmail.com|     Pune|9876543213|2024-03-15|Inactive|
|        105|  Karan Patel| karan@gmail.com|Ahmedabad|9876543214|2024-04-20|  Active|
|        106|  Sneha Joshi| sneha@gmail.com|   Indore|9876543215|2024-05-12|  Active|
|        107|  Rohan Mehta| rohan@gmail.com|  Lucknow|9876543216|2024-06-05|  Active|
|        108|Anjali Sharma|anjali@gmail.com|   Bhopal|9876543217|2024-06-20|Inactive|
|        109|   Vikas Jain| vikas@gmail.com|     Kota|

In [0]:
print("Total Records :", master_df.count())

Total Records : 20


In [0]:
print("Total Records :", master_df.count())

print("Distinct Records :", master_df.distinct().count())

Total Records : 20
Distinct Records : 20


In [0]:
clean_df = master_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import col, when, count
clean_df.select(
[count(
when(col(c).isNull(), c)
).alias(c)
for c in clean_df.columns]).show()

+-----------+-------------+-----+----+-----+---------+------+
|Customer_ID|Customer_Name|Email|City|Phone|Join_Date|Status|
+-----------+-------------+-----+----+-----+---------+------+
|          0|            0|    0|   0|    0|        0|     0|
+-----------+-------------+-----+----+-----+---------+------+



In [0]:
clean_df.printSchema()

root
 |-- Customer_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Status: string (nullable = true)



In [0]:
clean_df.show(10)

+-----------+-------------+----------------+--------+----------+----------+--------+
|Customer_ID|Customer_Name|           Email|    City|     Phone| Join_Date|  Status|
+-----------+-------------+----------------+--------+----------+----------+--------+
|        107|  Rohan Mehta| rohan@gmail.com| Lucknow|9876543216|2024-06-05|  Active|
|        102| Rahul Sharma| rahul@gmail.com|   Delhi|9876543211|2024-02-01|  Active|
|        103|  Priya Gupta| priya@gmail.com|  Mumbai|9876543212|2024-03-10|  Active|
|        112|  Komal Verma| komal@gmail.com|   Delhi|9876543221|2024-08-18|  Active|
|        114|  Nisha Patel| nisha@gmail.com|   Surat|9876543223|2024-09-10|Inactive|
|        117| Manish Gupta|manish@gmail.com|   Noida|9876543226|2024-11-01|  Active|
|        111| Ritesh Kumar|ritesh@gmail.com|   Patna|9876543220|2024-08-02|  Active|
|        116|  Simran Kaur|simran@gmail.com|Amritsar|9876543225|2024-10-15|  Active|
|        101|   Amit Singh|  amit@gmail.com|  Jaipur|9876543210|2

In [0]:
delta_df.show()

+-----------+-------------+------------------+----------+----------+----------+--------+
|Customer_ID|Customer_Name|             Email|      City|     Phone| Join_Date|  Status|
+-----------+-------------+------------------+----------+----------+----------+--------+
|        101|   Amit Singh|    amit@gmail.com|    Jaipur|9876543210|2024-01-15|  Active|
|        102| Rahul Sharma|   rahul@gmail.com|     Delhi|9876543211|2024-02-01|  Active|
|        103|  Priya Gupta|   priya@gmail.com|    Mumbai|9876543212|2024-03-10|  Active|
|        104|   Neha Verma|    neha@gmail.com|      Pune|9876543213|2024-03-15|Inactive|
|        105|  Karan Patel|   karan@gmail.com| Ahmedabad|9876543214|2024-04-20|  Active|
|        106|  Sneha Joshi|   sneha@gmail.com|    Indore|9876543215|2024-05-12|  Active|
|        107|  Rohan Mehta|   rohan@gmail.com|   Lucknow|9876543216|2024-06-05|  Active|
|        108|Anjali Sharma|  anjali@gmail.com|    Bhopal|9876543217|2024-06-20|Inactive|
|        109|   Vikas

# Incremental Data Loading using MERGE

In [0]:
increment_df.show()

+-----------+---------------+-------------------+---------+----------+----------+--------+
|Customer_ID|  Customer_Name|              Email|     City|     Phone| Join_Date|  Status|
+-----------+---------------+-------------------+---------+----------+----------+--------+
|        102|   Rahul Sharma|    rahul@gmail.com|    Noida|9876543211|2024-02-01|  Active|
|        104|     Neha Verma|     neha@gmail.com|Hyderabad|9876543213|2024-03-15|  Active|
|        115|   Deepak Yadav|   deepak@gmail.com|  Gurgaon|9876543224|2024-10-05|  Active|
|        121|    Sahil Verma|    sahil@gmail.com|     Pune|9876543230|2025-01-05|  Active|
|        122|    Ayesha Khan|   ayesha@gmail.com|   Mumbai|9876543231|2025-01-10|  Active|
|        123|   Mohit Sharma|    mohit@gmail.com|   Jaipur|9876543232|2025-01-15|  Active|
|        124|Priyanshi Gupta|priyanshi@gmail.com|    Delhi|9876543233|2025-01-20|Inactive|
|        125|    Harsh Patel|    harsh@gmail.com|Ahmedabad|9876543234|2025-02-01|  Active|

In [0]:
print("Incremental Records :", increment_df.count())

Incremental Records : 8


In [0]:
delta_df.show()

+-----------+-------------+------------------+----------+----------+----------+--------+
|Customer_ID|Customer_Name|             Email|      City|     Phone| Join_Date|  Status|
+-----------+-------------+------------------+----------+----------+----------+--------+
|        101|   Amit Singh|    amit@gmail.com|    Jaipur|9876543210|2024-01-15|  Active|
|        102| Rahul Sharma|   rahul@gmail.com|     Delhi|9876543211|2024-02-01|  Active|
|        103|  Priya Gupta|   priya@gmail.com|    Mumbai|9876543212|2024-03-10|  Active|
|        104|   Neha Verma|    neha@gmail.com|      Pune|9876543213|2024-03-15|Inactive|
|        105|  Karan Patel|   karan@gmail.com| Ahmedabad|9876543214|2024-04-20|  Active|
|        106|  Sneha Joshi|   sneha@gmail.com|    Indore|9876543215|2024-05-12|  Active|
|        107|  Rohan Mehta|   rohan@gmail.com|   Lucknow|9876543216|2024-06-05|  Active|
|        108|Anjali Sharma|  anjali@gmail.com|    Bhopal|9876543217|2024-06-20|Inactive|
|        109|   Vikas

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(
spark,
"/Volumes/workspace/default/delta_lake/customer_delta"
)

In [0]:
delta_table.alias("target").merge(
increment_df.alias("source"),
"target.Customer_ID = source.Customer_ID"
).whenMatchedUpdate(
set={
"Customer_Name":"source.Customer_Name",
"Email":"source.Email",
"City":"source.City",
"Phone":"source.Phone",
"Join_Date":"source.Join_Date",
"Status":"source.Status"
}
).whenNotMatchedInsert(
values={
"Customer_ID":"source.Customer_ID",
"Customer_Name":"source.Customer_Name",
"Email":"source.Email",
"City":"source.City",
"Phone":"source.Phone",
"Join_Date":"source.Join_Date",
"Status":"source.Status"
}
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

The MERGE operation compares the **Customer_ID** in the source and target datasets.
If the customer already exists:
- The existing record is updated.
If the customer does not exist:
- A new record is inserted.|
This allows incremental data loading without replacing the complete dataset.

In [0]:
updated_df = spark.read.format("delta").load(
"/Volumes/workspace/default/delta_lake/customer_delta"

)

In [0]:
updated_df.show(30)

+-----------+---------------+-------------------+----------+----------+----------+--------+
|Customer_ID|  Customer_Name|              Email|      City|     Phone| Join_Date|  Status|
+-----------+---------------+-------------------+----------+----------+----------+--------+
|        101|     Amit Singh|     amit@gmail.com|    Jaipur|9876543210|2024-01-15|  Active|
|        103|    Priya Gupta|    priya@gmail.com|    Mumbai|9876543212|2024-03-10|  Active|
|        105|    Karan Patel|    karan@gmail.com| Ahmedabad|9876543214|2024-04-20|  Active|
|        106|    Sneha Joshi|    sneha@gmail.com|    Indore|9876543215|2024-05-12|  Active|
|        107|    Rohan Mehta|    rohan@gmail.com|   Lucknow|9876543216|2024-06-05|  Active|
|        108|  Anjali Sharma|   anjali@gmail.com|    Bhopal|9876543217|2024-06-20|Inactive|
|        109|     Vikas Jain|    vikas@gmail.com|      Kota|9876543218|2024-07-01|  Active|
|        110|    Pooja Singh|    pooja@gmail.com|   Udaipur|9876543219|2024-07-1

In [0]:
print("Total Customers After Merge :", updated_df.count())

Total Customers After Merge : 25


In [0]:
updated_df.filter(
col("Customer_ID")==102
).show()

+-----------+-------------+---------------+-----+----------+----------+------+
|Customer_ID|Customer_Name|          Email| City|     Phone| Join_Date|Status|
+-----------+-------------+---------------+-----+----------+----------+------+
|        102| Rahul Sharma|rahul@gmail.com|Noida|9876543211|2024-02-01|Active|
+-----------+-------------+---------------+-----+----------+----------+------+



In [0]:
updated_df.filter(
col("Customer_ID")==104
).show()

+-----------+-------------+--------------+---------+----------+----------+------+
|Customer_ID|Customer_Name|         Email|     City|     Phone| Join_Date|Status|
+-----------+-------------+--------------+---------+----------+----------+------+
|        104|   Neha Verma|neha@gmail.com|Hyderabad|9876543213|2024-03-15|Active|
+-----------+-------------+--------------+---------+----------+----------+------+



In [0]:
updated_df.filter(
col("Customer_ID")==115
).show()

+-----------+-------------+----------------+-------+----------+----------+------+
|Customer_ID|Customer_Name|           Email|   City|     Phone| Join_Date|Status|
+-----------+-------------+----------------+-------+----------+----------+------+
|        115| Deepak Yadav|deepak@gmail.com|Gurgaon|9876543224|2024-10-05|Active|
+-----------+-------------+----------------+-------+----------+----------+------+



In [0]:
updated_df.filter(
col("Customer_ID")>=121
).show()

+-----------+---------------+-------------------+---------+----------+----------+--------+
|Customer_ID|  Customer_Name|              Email|     City|     Phone| Join_Date|  Status|
+-----------+---------------+-------------------+---------+----------+----------+--------+
|        121|    Sahil Verma|    sahil@gmail.com|     Pune|9876543230|2025-01-05|  Active|
|        122|    Ayesha Khan|   ayesha@gmail.com|   Mumbai|9876543231|2025-01-10|  Active|
|        123|   Mohit Sharma|    mohit@gmail.com|   Jaipur|9876543232|2025-01-15|  Active|
|        124|Priyanshi Gupta|priyanshi@gmail.com|    Delhi|9876543233|2025-01-20|Inactive|
|        125|    Harsh Patel|    harsh@gmail.com|Ahmedabad|9876543234|2025-02-01|  Active|
+-----------+---------------+-------------------+---------+----------+----------+--------+



In [0]:
print(
"Distinct Customers :",
updated_df.select(
"Customer_ID"
).distinct().count()
)

Distinct Customers : 25


In [0]:
updated_df.printSchema()

root
 |-- Customer_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Status: string (nullable = true)



In [0]:
updated_df.orderBy(
"Customer_ID"
).show(30)

+-----------+---------------+-------------------+----------+----------+----------+--------+
|Customer_ID|  Customer_Name|              Email|      City|     Phone| Join_Date|  Status|
+-----------+---------------+-------------------+----------+----------+----------+--------+
|        101|     Amit Singh|     amit@gmail.com|    Jaipur|9876543210|2024-01-15|  Active|
|        102|   Rahul Sharma|    rahul@gmail.com|     Noida|9876543211|2024-02-01|  Active|
|        103|    Priya Gupta|    priya@gmail.com|    Mumbai|9876543212|2024-03-10|  Active|
|        104|     Neha Verma|     neha@gmail.com| Hyderabad|9876543213|2024-03-15|  Active|
|        105|    Karan Patel|    karan@gmail.com| Ahmedabad|9876543214|2024-04-20|  Active|
|        106|    Sneha Joshi|    sneha@gmail.com|    Indore|9876543215|2024-05-12|  Active|
|        107|    Rohan Mehta|    rohan@gmail.com|   Lucknow|9876543216|2024-06-05|  Active|
|        108|  Anjali Sharma|   anjali@gmail.com|    Bhopal|9876543217|2024-06-2

# Observations
The following tasks were successfully completed:
- Loaded the incremental customer dataset.
- Converted the master dataset into a Delta Table.
- Applied the MERGE (UPSERT) operation.
- Updated existing customer records.
- Inserted new customer records.
- Verified the updated Delta table.
- Validated record counts and data consistency.
The MERGE operation efficiently synchronized the master dataset with the latest incremental data.

In [0]:
delta_table.history().show(truncate=False)

+-------+-------------------+--------------+--------------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------